In [11]:

import websocket
import json
import csv
import datetime
import os
import pandas as pd
import threading
import time
from sqlalchemy import create_engine
import urllib.parse

In [12]:

WEBSOCKET_URL = "wss://ws.bitget.com/v2/ws/public"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
CSV_FILE_NAME = "data.csv"

In [13]:

def tao_file_csv():
    header = ["thoi_gian", "gia", "gia_mua", "gia_ban", "khoi_luong_24h","high24h", "low24h","instId","best_purchase_price", "best_sale_price"]
    
    if not os.path.exists(CSV_FILE_NAME) or os.path.getsize(CSV_FILE_NAME) == 0:
        with open(CSV_FILE_NAME, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV: {CSV_FILE_NAME}")

tao_file_csv()

Đã tạo file CSV: data.csv


In [14]:

def on_open(ws):
    print(f" Đã kết nối thành công")
    
    
    for inst_id in INSTRUMENT_IDS:
        subscribe_message = {
            "op": "subscribe",
            "args": [
                {    
                    "instType": "USDT-FUTURES",
                    "channel": "ticker",
                    "instId": inst_id
                }
            ]
        }
        ws.send(json.dumps(subscribe_message))
        print(f"Đang theo dõi {inst_id}")
    print(f"Đang theo dõi các cặp: {', '.join(INSTRUMENT_IDS)}")

all_data= []

def on_message(ws, message_str):
    global all_data
    data = json.loads(message_str)
    all_data.append(data)
    
    if "data" in data and data["data"]:
        ticker = data["data"][0]
        
        
        thoi_gian = datetime.datetime.now().isoformat()
        gia = ticker.get('lastPr')
        gia_mua = ticker.get('bidPr')
        gia_ban = ticker.get('askPr')
        khoi_luong = ticker.get('volumeUsd24h')
        high_int_24h= ticker.get('high24h')
        low_int_24h = ticker.get('low24h')
        instId = ticker.get('instId') 
        best_purchase_price= ticker.get('bidSz')
        best_sale_price = ticker.get('askSz')
        
        
        # Hiển thị
        print(f" {instId} | Giá: {gia}") 
        
        # Lưu vào CSV
        with open(CSV_FILE_NAME, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([thoi_gian, gia, gia_mua, gia_ban, khoi_luong, high_int_24h, low_int_24h, instId, best_purchase_price, best_sale_price])

def on_error(ws, error):
    print(f" Lỗi: {error}")

def on_close(ws, close_status_code, close_msg):
    print(f" Kết nối đã đóng")

In [15]:
a=60  #1 phút
b=a*60  #1h
c=b*24  #1 ngày
def run_ws():
    ws.run_forever(ping_interval=30, ping_timeout=10)
ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open,
                          on_message=on_message,
                          on_error=on_error,
                          on_close=on_close)

print(f"Bắt đầu kết nối đến Bitget...")
print(f"Dữ liệu sẽ được lưu vào: {CSV_FILE_NAME}")
print("Nhấn Ctrl+C để dừng")

ws_thread = threading.Thread(target=run_ws)
ws_thread.daemon = True
ws_thread.start()

run_duration = 10

try:
    time.sleep(run_duration)
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")


ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")

# nhìn time ở trên tự tính hoặc để mặc định 
# add time ở phần Run_duration
# rồi chạy cell dataframe
# muốn xóa  phần add vào csv thì xóa phần CSV_FILE_NAME = "data.csv" ở trên


Bắt đầu kết nối đến Bitget...
Dữ liệu sẽ được lưu vào: data.csv
Nhấn Ctrl+C để dừng
 Đã kết nối thành công
Đang theo dõi SOLUSDT
Đang theo dõi BTCUSDT
Đang theo dõi ETHUSDT
Đang theo dõi các cặp: SOLUSDT, BTCUSDT, ETHUSDT
 BTCUSDT | Giá: 105477.5
 ETHUSDT | Giá: 2531.39
 SOLUSDT | Giá: 157.304
 BTCUSDT | Giá: 105477.5
 ETHUSDT | Giá: 2531.38
 SOLUSDT | Giá: 157.304
 ETHUSDT | Giá: 2531.39
 BTCUSDT | Giá: 105477.5
 SOLUSDT | Giá: 157.304
 BTCUSDT | Giá: 105477.5
 ETHUSDT | Giá: 2531.39
 SOLUSDT | Giá: 157.304
 ETHUSDT | Giá: 2531.39
 BTCUSDT | Giá: 105477.5
 SOLUSDT | Giá: 157.304
 BTCUSDT | Giá: 105477.5
 SOLUSDT | Giá: 157.304
 ETHUSDT | Giá: 2531.39
 BTCUSDT | Giá: 105463.9
 ETHUSDT | Giá: 2529.73
 SOLUSDT | Giá: 157.319
 BTCUSDT | Giá: 105463.9
 ETHUSDT | Giá: 2529.73
 SOLUSDT | Giá: 157.319
 BTCUSDT | Giá: 105461.1
 ETHUSDT | Giá: 2529.73
 SOLUSDT | Giá: 157.273
 BTCUSDT | Giá: 105461.2
 ETHUSDT | Giá: 2529.73
 SOLUSDT | Giá: 157.273
 BTCUSDT | Giá: 105461.2
 ETHUSDT | Giá: 2529.74

In [16]:

ws.close()


data_for_df = []
for msg in all_data:
    if "data" in msg and msg["data"]:
        ticker = msg["data"][0]
        
        row = {
            "thoi_gian": datetime.datetime.now().isoformat(), 
            "gia": ticker.get('lastPr'),
            "gia_mua": ticker.get('bidPr'),
            "gia_ban": ticker.get('askPr'),
            "khoi_luong_24h": ticker.get('volumeUsd24h'),
            "high24h": ticker.get('high24h'),
            "low24h": ticker.get('low24h'),
            "instId": ticker.get('instId'),
            "best_purchase_price": ticker.get('bidSz'),
            "best_sale_price": ticker.get('askSz')
        }
        data_for_df.append(row)


df = pd.DataFrame(data_for_df)



In [37]:
df

,thoi_gian,gia,gia_mua,gia_ban,khoi_luong_24h,high24h,low24h,instid,best_purchase_price,best_sale_price
0,2025-05-30 20:49:27.941637,2587.44,2587.43,2587.44,None,2674.55,2557.62,ETHUSDT,111.33,30.04
1,2025-05-30 20:49:27.941637,105700,105700,105700.1,None,107849.9,104560,BTCUSDT,29.6749,0.8232
2,2025-05-30 20:49:27.941637,161.88,161.899,161.9,None,171.381,159.9,SOLUSDT,64.9,369.8
3,2025-05-30 20:49:27.941637,2587.45,2587.44,2587.45,None,2674.55,2557.62,ETHUSDT,122.75,0.01
4,2025-05-30 20:49:27.941637,105710,105713.2,105713.5,None,107849.9,104560,BTCUSDT,28.3805,0.495
...,...,...,...,...,...,...,...,...,...,...
95,2025-05-30 21:11:46.974040,160.91,160.91,160.92,None,170.898,159.9,SOLUSDT,11,244
96,2025-05-30 21:11:46.974040,105539,105538.9,105539,None,107776.1,104560,BTCUSDT,24.8567,3.189
97,2025-05-30 21:11:46.974040,2579.94,2579.69,2579.7,None,2674.55,2557.62,ETHUSDT,65.39,1.04
98,2025-05-30 21:11:46.974040,160.94,160.939,160.94,None,170.898,159.9,SOLUSDT,153.5,220.1


In [17]:

host = 'localhost'  
port = '5000'  
database = 'postgres'  
username = 'postgres'
password = 'postgres'  


conn_url = f'postgresql://{username}:{urllib.parse.quote_plus(password)}@{host}:{port}/{database}'
engine = create_engine(conn_url)

try:
    # Thử kết nối trước khi làm việc với DataFrame
    with engine.connect() as connection:
        print("Kết nối PostgreSQL thành công!")
    
    df['thoi_gian'] = pd.to_datetime(df['thoi_gian'])
    
    
    df = df.rename(columns={'instId': 'instid'})
    
    # Lưu vào database
    df.to_sql('coin_prices', engine, if_exists='append', index=False, 
              method='multi', chunksize=1000)
    print(f"Đã lưu thành công {len(df)} dòng dữ liệu vào PostgreSQL")
except Exception as e:
    print(f"Lỗi khi lưu dữ liệu: {e}")
#bảng    
# CREATE TABLE coin_prices (
#     id SERIAL PRIMARY KEY,
#     thoi_gian TIMESTAMPTZ NOT NULL,
#     gia NUMERIC(20, 8),
#     gia_mua NUMERIC(20, 8),
#     gia_ban NUMERIC(20, 8),
#     khoi_luong_24h NUMERIC(30, 8),
#     high24h NUMERIC(20, 8),
#     low24h NUMERIC(20, 8),
#     instId TEXT,  
#     best_purchase_price NUMERIC(20, 8),
#     best_sale_price NUMERIC(20, 8)
# );

Kết nối PostgreSQL thành công!
Đã lưu thành công 72 dòng dữ liệu vào PostgreSQL


-- Xem toàn bộ dữ liệu (có giới hạn)
SELECT * FROM coin_prices LIMIT 100;



In [25]:
query = "SELECT * FROM coin_prices LIMIT 100"
pd.read_sql(query, engine)

,thoi_gian,gia,gia_mua,gia_ban,khoi_luong_24h,high24h,low24h,instid,best_purchase_price,best_sale_price
0,2025-05-30 20:49:27.941637,2587.44,2587.43,2587.44,None,2674.55,2557.62,ETHUSDT,111.33,30.04
1,2025-05-30 20:49:27.941637,105700,105700,105700.1,None,107849.9,104560,BTCUSDT,29.6749,0.8232
2,2025-05-30 20:49:27.941637,161.88,161.899,161.9,None,171.381,159.9,SOLUSDT,64.9,369.8
3,2025-05-30 20:49:27.941637,2587.45,2587.44,2587.45,None,2674.55,2557.62,ETHUSDT,122.75,0.01
4,2025-05-30 20:49:27.941637,105710,105713.2,105713.5,None,107849.9,104560,BTCUSDT,28.3805,0.495
...,...,...,...,...,...,...,...,...,...,...
95,2025-05-30 21:11:46.974040,160.91,160.91,160.92,None,170.898,159.9,SOLUSDT,11,244
96,2025-05-30 21:11:46.974040,105539,105538.9,105539,None,107776.1,104560,BTCUSDT,24.8567,3.189
97,2025-05-30 21:11:46.974040,2579.94,2579.69,2579.7,None,2674.55,2557.62,ETHUSDT,65.39,1.04
98,2025-05-30 21:11:46.974040,160.94,160.939,160.94,None,170.898,159.9,SOLUSDT,153.5,220.1


In [33]:
# Kiểm tra kết nối PostgreSQL
import socket
import subprocess
import time

def check_port_open(host, port):
    """Kiểm tra xem port có đang mở không"""
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = sock.connect_ex((host, int(port)))
    sock.close()
    return result == 0

# Kiểm tra port
print(f"Kiểm tra port {port} trên {host}:", "Đang mở" if check_port_open(host, port) else "Đóng")

# Kiểm tra Docker container
try:
    result = subprocess.run(["docker", "ps"], capture_output=True, text=True)
    if "postgres_container" in result.stdout:
        print("PostgreSQL container đang chạy")
    else:
        print("PostgreSQL container có thể chưa chạy")
        print("Hãy chạy lệnh: cd \"c:\\Dự án tốt nghiêp\\crwaling_data\\postgre-compose\" && docker-compose up -d")
except Exception as e:
    print(f"Không thể kiểm tra Docker: {e}")

# Thử kết nối trực tiếp với PostgreSQL
try:
    import psycopg2
    conn = psycopg2.connect(
        host=host,
        port=port,
        dbname=database,
        user=username,
        password=password
    )
    conn.close()
    print("Kết nối PostgreSQL thành công!")
except Exception as e:
    print(f"Lỗi kết nối PostgreSQL: {e}")
    
# Các bước xử lý lỗi
print("\nCác bước kiểm tra nếu vẫn gặp lỗi:")
print("1. Đảm bảo container PostgreSQL đang chạy: docker-compose up -d")
print("2. Đảm bảo port 5433 không bị chặn bởi tường lửa")
print("3. Kiểm tra lại thông tin kết nối trong pgAdmin:")
print("   - Host: localhost")
print(f"   - Port: {port}")
print("   - Username: postgres")
print("   - Password: postgres")

Kiểm tra port 5000 trên localhost: Đang mở
PostgreSQL container đang chạy
Kết nối PostgreSQL thành công!

Các bước kiểm tra nếu vẫn gặp lỗi:
1. Đảm bảo container PostgreSQL đang chạy: docker-compose up -d
2. Đảm bảo port 5433 không bị chặn bởi tường lửa
3. Kiểm tra lại thông tin kết nối trong pgAdmin:
   - Host: localhost
   - Port: 5000
   - Username: postgres
   - Password: postgres


# Cấu hình và Kiểm tra PostgreSQL

Cell này chứa mã để tạo bảng `coin_prices` trong PostgreSQL nếu chưa tồn tại. 
Chạy cell này trước khi thử lưu dữ liệu để đảm bảo bảng đã được tạo đúng cách.

In [5]:
# Chi tiết kiểm tra kết nối PostgreSQL
import socket
import subprocess
import os

def run_docker_command(command):
    """Thực thi lệnh docker và trả về kết quả"""
    try:
        result = subprocess.run(command, capture_output=True, text=True, shell=True)
        return result.stdout
    except Exception as e:
        return f"Lỗi: {e}"

# Kiểm tra thông tin hệ thống
print("=== THÔNG TIN HỆ THỐNG ===")
print(f"- Host: {host}")
print(f"- Port: {port}")
print(f"- Database: {database}")
print(f"- Username: {username}")

# Kiểm tra kết nối tới port
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
result = sock.connect_ex((host, int(port)))
sock.close()
print(f"\n=== KIỂM TRA KẾT NỐI ===")
print(f"- Kết nối tới {host}:{port}: {'Thành công' if result == 0 else 'Thất bại'}")

# Kiểm tra container
print("\n=== THÔNG TIN DOCKER ===")
docker_ps = run_docker_command("docker ps | findstr postgres")
print(f"- Docker container status: \n{docker_ps}")

# Kiểm tra logs của container
print("\n=== LOGS GẦN NHẤT CỦA CONTAINER ===")
docker_logs = run_docker_command("docker logs postgres_container --tail 10")
print(docker_logs)

# Thử kết nối trực tiếp PostgreSQL bằng cả localhost và 127.0.0.1
print("\n=== THỬ KẾT NỐI TRỰC TIẾP ===")
for test_host in ['localhost', '127.0.0.1']:
    try:
        import psycopg2
        conn_string = f"host={test_host} port={port} dbname={database} user={username} password={password}"
        conn = psycopg2.connect(conn_string)
        conn.close()
        print(f"- Kết nối tới {test_host}:{port}: ✅ Thành công")
    except Exception as e:
        print(f"- Kết nối tới {test_host}:{port}: ❌ Thất bại - {e}")

# Kiểm tra truy vấn thử
print("\n=== THỬ TRUY VẤN POSTGRESQL ===")
try:
    # Thử với host thành công từ bước trên hoặc mặc định localhost
    conn = psycopg2.connect(
        host=host,
        port=port,
        dbname=database,
        user=username,
        password=password
    )
    cursor = conn.cursor()
    cursor.execute("SELECT version();")
    version = cursor.fetchone()[0]
    print(f"- PostgreSQL version: {version}")
    
    # Kiểm tra bảng coin_prices
    cursor.execute("SELECT EXISTS (SELECT FROM information_schema.tables WHERE table_name = 'coin_prices');")
    table_exists = cursor.fetchone()[0]
    print(f"- Bảng coin_prices tồn tại: {table_exists}")
    
    cursor.close()
    conn.close()
except Exception as e:
    print(f"- Thất bại khi truy vấn: {e}")

# Gợi ý xử lý
print("\n=== GỢI Ý XỬ LÝ ===")
print("1. Nếu kết nối localhost không hoạt động, thay đổi 'host' trong cell kết nối sang '127.0.0.1'")
print("2. Nếu vẫn gặp lỗi, thử khởi động lại container:")
print("   - cd \"c:\\Dự án tốt nghiêp\\crwaling_data\\postgre-compose\"")
print("   - docker-compose down")
print("   - docker-compose up -d")
print("3. Kiểm tra xem port 5433 có đang bị chiếm bởi ứng dụng khác không:")
print("   - netstat -ano | findstr 5433")

=== THÔNG TIN HỆ THỐNG ===
- Host: localhost
- Port: 5000
- Database: postgres
- Username: postgres

=== KIỂM TRA KẾT NỐI ===
- Kết nối tới localhost:5000: Thành công

=== THÔNG TIN DOCKER ===
- Docker container status: 
7956fe9b8e43   postgres:latest             "docker-entrypoint.s…"   2 minutes ago   Up 2 minutes   0.0.0.0:5000->5432/tcp          postgres_container


=== LOGS GẦN NHẤT CỦA CONTAINER ===

PostgreSQL Database directory appears to contain a database; Skipping initialization



=== THỬ KẾT NỐI TRỰC TIẾP ===
- Kết nối tới localhost:5000: ✅ Thành công
- Kết nối tới 127.0.0.1:5000: ✅ Thành công

=== THỬ TRUY VẤN POSTGRESQL ===
- PostgreSQL version: PostgreSQL 17.5 (Debian 17.5-1.pgdg120+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14) 12.2.0, 64-bit
- Bảng coin_prices tồn tại: True

=== GỢI Ý XỬ LÝ ===
1. Nếu kết nối localhost không hoạt động, thay đổi 'host' trong cell kết nối sang '127.0.0.1'
2. Nếu vẫn gặp lỗi, thử khởi động lại container:
   - cd "c:\Dự án 

In [36]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host='localhost',
    port='5000',
    database='postgres',
    user='postgres',
    password='postgres'
)

query = "SELECT * FROM coin_prices LIMIT 100"
pd.read_sql(query, engine)




,thoi_gian,gia,gia_mua,gia_ban,khoi_luong_24h,high24h,low24h,instid,best_purchase_price,best_sale_price
0,2025-05-30 20:49:27.941637,2587.44,2587.43,2587.44,None,2674.55,2557.62,ETHUSDT,111.33,30.04
1,2025-05-30 20:49:27.941637,105700,105700,105700.1,None,107849.9,104560,BTCUSDT,29.6749,0.8232
2,2025-05-30 20:49:27.941637,161.88,161.899,161.9,None,171.381,159.9,SOLUSDT,64.9,369.8
3,2025-05-30 20:49:27.941637,2587.45,2587.44,2587.45,None,2674.55,2557.62,ETHUSDT,122.75,0.01
4,2025-05-30 20:49:27.941637,105710,105713.2,105713.5,None,107849.9,104560,BTCUSDT,28.3805,0.495
...,...,...,...,...,...,...,...,...,...,...
95,2025-05-30 21:11:46.974040,160.91,160.91,160.92,None,170.898,159.9,SOLUSDT,11,244
96,2025-05-30 21:11:46.974040,105539,105538.9,105539,None,107776.1,104560,BTCUSDT,24.8567,3.189
97,2025-05-30 21:11:46.974040,2579.94,2579.69,2579.7,None,2674.55,2557.62,ETHUSDT,65.39,1.04
98,2025-05-30 21:11:46.974040,160.94,160.939,160.94,None,170.898,159.9,SOLUSDT,153.5,220.1
